# Create Fondation Leducq Awards

Creates Fondation Leducq (Leducq Foundation, Paris/Boston) awards from the
foundation's own website (WordPress, static HTML):

1. **Transatlantic / International Networks of Excellence, 2004-2026 (93 networks)** from
   https://www.fondationleducq.org/international-networks-of-excellence-how-to-apply/funded-networks/
   (every network by award year, title, European + North American coordinators; since
   2022 just "Coordinators") plus each network page `/network/<slug>/` (members,
   description). Lead investigator = European (or first-listed) coordinator, co-lead =
   North American (or second) coordinator; all coordinators + members in `investigators`.
   Where a coordinator died and was replaced ("A († date)/B"), the original awardee stays
   lead and the successor is in `investigators`.
2. **Equipement de Recherche et Plateformes Technologiques (ERPT), 2017 (11 grants)** from
   https://www.fondationleducq.org/retp/ — French research-equipment grants approved by the
   board on 17 May 2017, with project holder, institution and the EUR amount requested
   from the foundation. Research infrastructure money, so in scope; no filter applied.

Not included: the Rheumatic Heart Disease calls (no awardees published) and the
2000-2002 "Bourses" / 2007-2009 young-investigator / Career Development Award
programmes (no longer on the site).

**104 awards (93 networks + 11 ERPT), 2004-2026, 100% title/description/PI.**
`amount` is NULL for networks: Leducq does not publish per-network amounts (the
programme gives roughly $6-8M over five years). §6.7 amount check waived for that reason;
the 11 ERPT grants carry EUR amounts (10.6% of rows). `start_date` is 1 January of the
award year for networks (the site gives only the year) and 2017-05-17 (board approval)
for ERPT.

**`funder_award_id` (§2.1.1):** Leducq numbers networks `YYCVDNN` (e.g. `16CVD03`), the
form citing works write in acknowledgements. The site prints it for the 15 networks of
2023-2026. For 2004-2022, `scripts/local/leducq_to_s3.py` carries a crosswalk
(`NETWORK_IDS`, 62 networks) resolved from the 7,358 OpenAlex works with Leducq as a
funder: for each cited `YYCVDNN`, the network of that award year whose coordinators and
members author the citing works (accepted only at >=30% coverage, >=3x the runner-up,
>=2 works or 1 by a coordinator; it reproduces every site-published number it can score,
and the mapping is one-to-one). The other 16 networks and the 11 ERPT grants ship
synthetic keys `LEDUCQ-<year>-<slug>` / `LEDUCQ-ERPT-2017-NN`. **66 of the 77 numbered
networks collapse onto 203 existing priority-0 citation rows** (crossref / europepmc /
datacite shells) in CreateAwards dedup.

**Prerequisites:** run `scripts/local/leducq_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/leducq/leducq_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320320890`, Fondation Leducq,
FR; ror 01czwga19 and doi 10.13039/501100001674 read from `openalex.funders.funders`.

**Priority:** `491` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.leducq_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/leducq/leducq_projects.parquet`;

In [ ]:
%sql
SELECT programme, network_number_source, COUNT(*) as n
FROM openalex.awards.leducq_raw
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.leducq_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.leducq_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320890 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320890;

## Step 2: Create Leducq Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.leducq_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320890  -- Fondation Leducq
),

people AS (
    -- coordinators (lead first, co-lead second) then members, in the order the script wrote them.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(
                from_json(people, 'array<struct<name:string,given_name:string,family_name:string,institution:string,country:string,role:string>>'),
                p -> p.family_name IS NOT NULL AND TRIM(p.family_name) != ''
            ),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', NULLIF(TRIM(p.country), ''),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.leducq_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    NULLIF(TRIM(g.currency), '') as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.programme = 'Networks of Excellence' THEN 'research' ELSE 'infrastructure' END as funding_type,
    CASE WHEN g.programme = 'Networks of Excellence' THEN 'Transatlantic Networks of Excellence'
         ELSE g.programme END as funder_scheme,
    'leducq_website' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.leducq_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check: every network ships YYCVDNN or the synthetic LEDUCQ-<year>-<slug>;
-- ERPT ships LEDUCQ-ERPT-2017-NN. Expect bad_shape = 0 and duplicate ids = 0.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{2}CVD[0-9]{2}$' THEN 1 ELSE 0 END) as cvd_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^LEDUCQ-(ERPT-2017-[0-9]{2}|[0-9]{4}-[a-z0-9-]+)$' THEN 1 ELSE 0 END) as synthetic,
    SUM(CASE WHEN NOT (funder_award_id RLIKE '^[0-9]{2}CVD[0-9]{2}$'
                    OR funder_award_id RLIKE '^LEDUCQ-(ERPT-2017-[0-9]{2}|[0-9]{4}-[a-z0-9-]+)$') THEN 1 ELSE 0 END) as bad_shape,
    COUNT(*) - COUNT(DISTINCT id) as duplicate_ids
FROM openalex.awards.leducq_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'leducq_website' AND priority = 491;

-- Insert into openalex_awards_raw with priority.
-- Priority 491: direct-from-funder ingest of Fondation Leducq's own site.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 491 outranks the
-- priority-0 acknowledgement shells that share the YYCVDNN award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    491 as priority
FROM openalex.awards.leducq_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.leducq_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, co_lead_investigator.family_name as co_lead,
       size(investigators) as n_investigators
FROM openalex.awards.leducq_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.leducq_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.leducq_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.leducq_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.leducq_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 / §6.7 coverage (expect 100% title/description/start_date/PI; amount ~10.6% = ERPT only, EUR).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(co_lead_investigator.family_name) as has_co_lead,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.leducq_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs: expect ~66 of our awards to share an id with
-- ~203 priority-0 rows (these upgrade in CreateAwards dedup).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(DISTINCT c.id) as distinct_awards
FROM openalex.awards.openalex_awards_raw c
JOIN openalex.awards.leducq_awards t ON t.id = c.id
WHERE c.funder_id = 4320320890 AND c.priority = 0
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'leducq_website'
GROUP BY provenance, priority;